In [1]:
!pip install torch torchvision ultralytics pandas pillow matplotlib tqdm scikit-learn kaggle opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s eta 0:00:00


In [2]:
import os, json

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json = {
    'username': 'shruti483',
    'key': 'KGAT_e18981c94b1d5eb7616a3cd9ad2683c8'
}

with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)

os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('kaggle.json created')


kaggle.json created


In [3]:
!kaggle datasets download -d a2015003713/militaryaircraftdetectiondataset
import zipfile

with zipfile.ZipFile(
    "militaryaircraftdetectiondataset.zip",
    'r'
) as zip_ref:

    zip_ref.extractall("dataset_files")

print("Dataset extracted")

Dataset URL: https://www.kaggle.com/datasets/a2015003713/militaryaircraftdetectiondataset
License(s): unknown
100% 11.2G/11.2G [00:56<00:00, 213MB/s]

Dataset extracted


In [4]:
import os

print(os.listdir("dataset_files"))

['labels_with_split.csv', 'annotated_samples', 'dataset', 'crop']


In [5]:
import pandas as pd

df = pd.read_csv(
    "dataset_files/labels_with_split.csv"
)

df.head()

,filename,width,height,class,xmin,ymin,xmax,ymax,split
0,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,1380,1904,1522,2014,train
1,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,1809,1625,1958,1759,train
2,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,2400,1571,2532,1727,train
3,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,3935,1772,4100,1891,train
4,00010041af654d0b8e1e16c824fa9867,1360,2048,UH60,835,526,1233,741,train


In [15]:
import os
import shutil
from tqdm import tqdm

# OUTPUT DIRECTORY
output_dir = "aircraft_yolo"

# IMAGE ROOT
image_src = "dataset_files/dataset"

# ONLY 8 CLASSES
selected_classes = [
    "Rafale",
    "Su57",
    "F22",
    "F35",
    "Tejas",
    "Mig29",
    "Mirage2000",
    "F16"
]

# FILTER DATAFRAME
df = df[df["class"].isin(selected_classes)]

# CLASSES
classes = sorted(df["class"].unique())

# CLASS IDS
class_to_id = {
    cls: idx for idx, cls in enumerate(classes)
}

print(class_to_id)

# FIND ALL JPG IMAGES
all_images = {}

for root, dirs, files in os.walk(image_src):

    for file in files:

        if file.endswith(".jpg"):

            all_images[file] = os.path.join(root, file)

print("Total images found:", len(all_images))

# CREATE YOLO DATASET
for _, row in tqdm(df.iterrows(), total=len(df)):

    # IMPORTANT FIX
    filename = row["filename"] + ".jpg"

    label = row["class"]

    split = row["split"]

    width = row["width"]
    height = row["height"]

    xmin = row["xmin"]
    ymin = row["ymin"]
    xmax = row["xmax"]
    ymax = row["ymax"]

    # YOLO FORMAT
    x_center = ((xmin + xmax) / 2) / width

    y_center = ((ymin + ymax) / 2) / height

    bbox_width = (xmax - xmin) / width

    bbox_height = (ymax - ymin) / height

    # CREATE FOLDERS
    image_folder = os.path.join(
        output_dir,
        "images",
        split
    )

    label_folder = os.path.join(
        output_dir,
        "labels",
        split
    )

    os.makedirs(image_folder, exist_ok=True)

    os.makedirs(label_folder, exist_ok=True)

    # COPY IMAGE
    if filename in all_images:

        src_img = all_images[filename]

        dst_img = os.path.join(
            image_folder,
            filename
        )

        shutil.copy(src_img, dst_img)

        # CREATE LABEL
        txt_name = filename.replace(
            ".jpg",
            ".txt"
        )

        txt_path = os.path.join(
            label_folder,
            txt_name
        )

        class_id = class_to_id[label]

        with open(txt_path, "w") as f:

            f.write(
                f"{class_id} "
                f"{x_center} "
                f"{y_center} "
                f"{bbox_width} "
                f"{bbox_height}"
            )

print("YOLO dataset ready")

{'F16': 0, 'Rafale': 1, 'Su57': 2, 'Tejas': 3}
Total images found: 23143


100%|██████████| 3525/3525 [00:13<00:00, 263.59it/s]

YOLO dataset ready


In [16]:
print(os.listdir("aircraft_yolo/images"))

print(
    "Train:",
    len(os.listdir("aircraft_yolo/images/train"))
)

print(
    "Validation:",
    len(os.listdir("aircraft_yolo/images/validation"))
)

print(
    "Test:",
    len(os.listdir("aircraft_yolo/images/test"))
)

['validation', 'test', 'train', 'valid']
Train: 1373
Validation: 378
Test: 133


In [17]:
yaml_text = f"""
path: aircraft_yolo

train: images/train
val: images/validation
test: images/test

names:
"""

for idx, cls in enumerate(classes):

    yaml_text += f"  {idx}: {cls}\n"

with open("aircraft.yaml", "w") as f:

    f.write(yaml_text)

print("aircraft.yaml created")

aircraft.yaml created


In [18]:
from ultralytics import YOLO

# LOAD YOLOv8
model = YOLO("yolov8n.pt")

# TRAIN
model.train(
    data="aircraft.yaml",
    epochs=30,
    imgsz=640,
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=aircraft.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=F

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a348bbb92b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [31]:
import os

print(
    os.listdir("aircraft_classifier/train")
)

['F16', 'Su57', 'Rafale', 'Tejas']


In [32]:
import os

print(df.iloc[0])

label = df.iloc[0]['class']

filename = str(df.iloc[0]['filename'])

print("\nLABEL:", label)

print("FILENAME:", filename)

print("\nChecking paths:\n")

print(
    os.path.join(
        "dataset_files/crop",
        label
    )
)

print(
    os.listdir(
        os.path.join(
            "dataset_files/crop",
            label
        )
    )[:5]
)

filename    000106393cfe2343888c584e65fd2274
width                                   3520
height                                  2514
class                                    F16
xmin                                       0
ymin                                       0
xmax                                    1796
ymax                                    1451
split                                  train
Name: 5, dtype: object

LABEL: F16
FILENAME: 000106393cfe2343888c584e65fd2274

Checking paths:

dataset_files/crop/F16
['4566d3a226f58658aafa34b6a03b75d4_2.jpg', '9665818d65a724230fae1e1f3a2210fc_3.jpg', 'ab0804d6fa2bc30fa99460e18a011c4e_1.jpg', '3342363db51f8b4642483ec090d5e342_0.jpg', '9c920d04c06784f6614ed2713c1e0955_0.jpg']


In [33]:
import os
import shutil
import random
from tqdm import tqdm

random.seed(42)

source_dir = "dataset_files/crop"

output_dir = "aircraft_classifier"

# DELETE OLD
shutil.rmtree(
    output_dir,
    ignore_errors=True
)

classes = [
    "F16",
    "Su57",
    "Rafale",
    "Tejas"
]

for cls in classes:

    class_path = os.path.join(
        source_dir,
        cls
    )

    images = os.listdir(class_path)

    random.shuffle(images)

    total = len(images)

    train_split = int(0.8 * total)

    valid_split = int(0.1 * total)

    train_images = images[:train_split]

    valid_images = images[
        train_split:
        train_split + valid_split
    ]

    test_images = images[
        train_split + valid_split:
    ]

    splits = {

        "train": train_images,

        "validation": valid_images,

        "test": test_images
    }

    for split_name, split_images in splits.items():

        split_folder = os.path.join(

            output_dir,

            split_name,

            cls
        )

        os.makedirs(
            split_folder,
            exist_ok=True
        )

        for img_name in tqdm(split_images):

            src = os.path.join(
                class_path,
                img_name
            )

            dst = os.path.join(
                split_folder,
                img_name
            )

            shutil.copy(src, dst)

print("Classifier dataset ready")

100%|██████████| 17/17 [00:00<00:00, 881.80it/s]

Classifier dataset ready


In [34]:
import os

print(
    os.listdir(
        "aircraft_classifier/train"
    )
)

print(
    len(os.listdir(
        "aircraft_classifier/train/F16"
    ))
)

print(
    len(os.listdir(
        "aircraft_classifier/validation/F16"
    ))
)

['F16', 'Su57', 'Rafale', 'Tejas']
1648
206


In [35]:
import torch

from torchvision import datasets
from torchvision import transforms

from torch.utils.data import DataLoader

transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(

    "aircraft_classifier/train",

    transform=transform
)

valid_dataset = datasets.ImageFolder(

    "aircraft_classifier/validation",

    transform=transform
)

train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True
)

valid_loader = DataLoader(

    valid_dataset,

    batch_size=32
)

print(train_dataset.classes)

['F16', 'Rafale', 'Su57', 'Tejas']


In [36]:
import torch.nn as nn

from torchvision import models

device = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model_cls = models.efficientnet_b0(
    pretrained=True
)

num_classes = len(
    train_dataset.classes
)

model_cls.classifier[1] = nn.Linear(

    model_cls.classifier[1].in_features,

    num_classes
)

model_cls = model_cls.to(device)

print(model_cls)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 103MB/s]

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [37]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(

    model_cls.parameters(),

    lr=0.0003
)

In [38]:
epochs = 20

for epoch in range(epochs):

    model_cls.train()

    running_loss = 0

    correct = 0

    total = 0

    for images, labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_cls(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch {epoch+1}")

    print(f"Loss: {running_loss:.4f}")

    print(f"Accuracy: {accuracy:.2f}%")

Epoch 1
Loss: 67.6713
Accuracy: 71.97%
Epoch 2
Loss: 29.8331
Accuracy: 89.67%
Epoch 3
Loss: 18.2093
Accuracy: 93.68%
Epoch 4
Loss: 12.1987
Accuracy: 95.64%
Epoch 5
Loss: 8.0727
Accuracy: 96.98%
Epoch 6
Loss: 9.7719
Accuracy: 97.69%
Epoch 7
Loss: 7.2050
Accuracy: 97.37%
Epoch 8
Loss: 4.7989
Accuracy: 98.15%
Epoch 9
Loss: 2.9895
Accuracy: 99.04%
Epoch 10
Loss: 3.3266
Accuracy: 98.72%
Epoch 11
Loss: 3.6715
Accuracy: 98.51%
Epoch 12
Loss: 3.1732
Accuracy: 99.04%
Epoch 13
Loss: 3.8435
Accuracy: 98.83%
Epoch 14
Loss: 3.8070
Accuracy: 98.58%
Epoch 15
Loss: 2.8088
Accuracy: 98.86%
Epoch 16
Loss: 3.7091
Accuracy: 99.01%
Epoch 17
Loss: 9.7514
Accuracy: 98.47%
Epoch 18
Loss: 8.8184
Accuracy: 96.56%
Epoch 19
Loss: 3.4239
Accuracy: 98.86%
Epoch 20
Loss: 1.9060
Accuracy: 99.40%


In [ ]:
epochs = 15

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================

    model_cls.train()

    running_loss = 0

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_cls(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    train_accuracy = 100 * correct / total

    # =========================
    # VALIDATION
    # =========================

    model_cls.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_cls(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()

    val_accuracy = 100 * val_correct / val_total

    # =========================
    # PRINT RESULTS
    # =========================

    print(f"\nEpoch {epoch+1}")

    print(f"Train Loss: {running_loss:.4f}")

    print(f"Train Accuracy: {train_accuracy:.2f}%")

    print(f"Validation Accuracy: {val_accuracy:.2f}%")

In [39]:
torch.save(

    model_cls.state_dict(),

    "efficientnet_aircraft.pth"
)

print("Classifier saved")

Classifier saved


In [55]:
from google.colab import files

uploaded = files.upload()

Saving t.webp to t.webp


In [57]:
from ultralytics import YOLO

detector = YOLO(
    "runs/detect/train/weights/best.pt"
)

results = detector(
    "t.webp"
)

results[0].save("detected.jpg")

print(results)


image 1/1 /content/t.webp: 352x640 1 Su57, 163.4ms
Speed: 7.8ms preprocess, 163.4ms inference, 2.0ms postprocess per image at shape (1, 3, 352, 640)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'F16', 1: 'Rafale', 2: 'Su57', 3: 'Tejas'}
obb: None
orig_img: array([[[255, 186, 168],
        [255, 186, 168],
        [255, 186, 168],
        ...,
        [229, 205, 180],
        [229, 205, 180],
        [229, 205, 180]],

       [[255, 186, 168],
        [255, 186, 168],
        [255, 186, 168],
        ...,
        [229, 205, 180],
        [229, 205, 180],
        [229, 205, 180]],

       [[255, 186, 168],
        [255, 186, 168],
        [255, 186, 168],
        ...,
        [229, 205, 180],
        [229, 205, 180],
        [229, 205, 180]],

       ...,

       [[224, 156, 135],
        [224, 156, 135],
        [224, 156, 135],
        ...,
        [202, 171, 149],
        [203, 172, 

In [58]:
import cv2

image = cv2.imread("t.webp")

boxes = results[0].boxes.xyxy.cpu().numpy()

x1, y1, x2, y2 = boxes[0]

x1 = int(x1)
y1 = int(y1)
x2 = int(x2)
y2 = int(y2)

crop = image[y1:y2, x1:x2]

cv2.imwrite(
    "cropped_aircraft.jpg",
    crop
)

print("Aircraft cropped")

Aircraft cropped


In [59]:
import torch

from torchvision import models

import torch.nn as nn

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

class_names = train_dataset.classes

model_cls = models.efficientnet_b0(
    pretrained=False
)

model_cls.classifier[1] = nn.Linear(
    model_cls.classifier[1].in_features,
    len(class_names)
)

model_cls.load_state_dict(
    torch.load(
        "efficientnet_aircraft.pth",
        map_location=device
    )
)

model_cls = model_cls.to(device)

model_cls.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [60]:
from PIL import Image

transform_test = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor()
])

img = Image.open(
    "cropped_aircraft.jpg"
).convert("RGB")

img = transform_test(img)

img = img.unsqueeze(0).to(device)

with torch.no_grad():

    outputs = model_cls(img)

    _, pred = torch.max(outputs, 1)

predicted_class = class_names[
    pred.item()
]

print(
    "Predicted Aircraft:",
    predicted_class
)

Predicted Aircraft: Tejas


In [61]:
torch.save(
    model_cls.state_dict(),
    "aircraft_classifier_efficientnet.pth"
)

print("Model saved successfully")

Model saved successfully


In [62]:
from google.colab import files

files.download(
    "/content/aircraft_classifier_efficientnet.pth"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
files.download(
    "/content/runs/detect/train/weights/best.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
import json

class_names = train_dataset.classes

with open("class_names.json", "w") as f:

    json.dump(class_names, f)

print("class_names.json saved")

class_names.json saved


In [66]:
files.download(
    "/content/class_names.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>